# 05 · Error analysis and export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/05_report.ipynb)

Show an item the model got wrong, and say whose fault it was.

```
  01_build_pool_<track>  →  02_sample  →  03_annotate  →  04_prompt  →▶ 05_report
```

| | |
|---|---|
| **Reads** | the gold set (03) · the frozen predictions and rounds (04) |
| **Writes** | `outputs/` — the predictions CSV, the report scaffold, a copy of your gold set |

---

This is the highest-value part of the whole project, and the one the Q&A will definitely go to. A low F1 with a clear account of *why* is worth more than a high one without.

## Setup — run this first

In Colab, uncomment **one** of the two clone blocks below before running. Colab starts with only this one file; the clone fetches everything *around* it (`scripts/`, `config.py`, `data/`) so the paths resolve.

**Do Option A once, as a group** — then always open the copy in Drive (*File ▸ Open ▸ Drive ▸ `lda2-final-template/notebooks/...`*). Your prompts, gold set and outputs then survive the runtime resetting, and everyone sees the same files.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first.
# ------------------------------------------------------------------
# In Google Colab, UNCOMMENT one of the two blocks below, then run the cell.

# --- Colab Option A: clone into your Google Drive (persists; do this once) ---
# from google.colab import drive
# drive.mount("/content/drive")
# %cd /content/drive/MyDrive
# ![ -d lda2-final-template ] || git clone https://github.com/egumasa/lda2-final-template.git lda2-final-template
# %cd /content/drive/MyDrive/lda2-final-template/notebooks

# --- Colab Option B: quick, throwaway clone (changes lost on reset) ---
# !git clone https://github.com/egumasa/lda2-final-template.git
# %cd lda2-final-template/notebooks

# Put scripts/ and config.py on the import path. Works locally AND in Colab
# after the %cd above, because notebooks/ sits beside both.
import sys
sys.path.append("../scripts")
sys.path.append("..")

from config import *      # TRACK, GROUP, SEED, N_PER_CLASS, and every path

from pipeline import *      # load_gold, load_predictions, load_json, ...
from metrics import *       # evaluate, show_errors

describe()                  # what this notebook is working on


> **Everything above comes from `config.py`** — one file at the top of the repo, which you edit once as a group. That is deliberate: the seed in notebook 02 has to be the seed in notebook 03, and five copies of a number in five notebooks is five chances for them to disagree. If the line it just printed is not your track, your group and your seed, fix `config.py` and re-run this cell.

In [ ]:
# ══ STEP 1 · Load the frozen run ══════════════════════════════════════════
# Goal      : the gold set, the predictions file, and the per-round table.
# Available : load_gold(GOLD_PATH)  ->  gold
#             load_predictions(PRED_PATH)  ->  pred_final
#             load_json(ROUNDS_PATH, what="rounds")  ->  f1_by_round
# Pointer   : Day 2 S6 — loading a frozen predictions file is exactly what you did there.
# Produce   : gold · pred_final · f1_by_round      ← later cells use these names
# Note      : nothing in this notebook calls the model. If a number here
#             differs from notebook 04, you are loading a different file —
#             not watching the model change its mind.

# ✏️ your code here


In [ ]:
# ══ STEP 2 · Score the frozen run ═════════════════════════════════════════
# Goal      : the headline numbers, from the file rather than from a live run.
# Available : evaluate(gold, pred_final, ordered=..., labels=LABELS_ORDER)  ->  macro-F1
# Pointer   : Day 2 S6 Part B · Day 3 — the identical call.
# Produce   : the numbers for report section 3      ← later cells use these names
# Note      : evaluate prints per-class P/R/F1 and kappa as well as the
#             macro average. "Which class is it worst at" is a more useful
#             sentence than "F1 = .62" — read the table, not just the
#             headline.
# Careful   : `agreement()` is for comparing two ANNOTATORS (two lists of
#             labels), not gold against predictions. You used it in 03.

# ✏️ your code here


## Step 3 — Error analysis

`show_errors` gives you every item the model got wrong. Two very different findings live in that table, and **your job is to say which is which**:

- **The model's fault** — the label is clear, both your annotators agreed on it immediately, and the model still missed it.
- **The scheme's fault** — the item is genuinely borderline. You know exactly which ones these are, because they are the rows you argued about in notebook 03.

So cross-reference: pull up your `disagreements` list from notebook 03 and see how much of it turns up here. **Overlap is a finding, not a failure.** If the model's errors cluster on the items your own annotators could not agree on, you have measured something real about the annotation scheme — and that is a better result than a clean F1.

You could not make this argument without having built the gold set yourselves. That is why notebook 03 exists.

In [ ]:
# ══ STEP 3 · Error analysis ═══════════════════════════════════════════════
# Goal      : find the misses, and attribute each one.
# Available : show_errors(gold, pred_final)  ->  a table of the items it got wrong
# Pointer   : Day 3, "where is your best prompt still wrong?" — identical call.
# Produce   : errors      ← later cells use these names
# Note      : pick at least three and write a REASON for each — model's
#             fault or scheme's fault, and how you know. Report section 4.
# Try       : errors.head(15)  ·  errors[errors.gold == "B2"]
# Ask       : how many of these ids are in your notebook-03
#             disagreements? That number is worth reporting.

# ✏️ your code here


## Step 4 — Export

Writes your gold set, a per-item predictions CSV, and a one-page report scaffold with the five required sections, all stamped with your group name.

The scaffold fills in what it can compute — labels, counts, the F1-per-round table. The *italic* placeholders are yours: the QC narrative, the error attributions, and limitations that apply to **your** run rather than the generic three. A section left as the placeholder scores zero, so this is the start of the writing, not the end.

In [ ]:
# ══ STEP 4 · Export ═══════════════════════════════════════════════════════
# Goal      : write the gold set, the predictions CSV, and the report scaffold.
# Available : export_results(TRACK, gold, pred_final, f1_by_round, OUT_DIR, group=GROUP)
# Pointer   : new — but it only writes down what you already have.
# Produce   : three files in ../outputs/      ← later cells use these names

# ✏️ your code here


---

## Hand it in

One command collects everything into a folder next to the repo, keeping the `scripts/ · prompts/ · data/ · notebooks/ · outputs/` layout — because that layout *is* the reproducibility checklist from S10, and because the notebooks' paths only resolve if it stays intact.

```bash
python scripts/make_submission.py --group groupA
```

It deliberately leaves out `.git/`, `.venv/`, your `.env` (**it holds your API key**), the big pools in `data/pools/`, and anything ICNALE-derived.

Then: find the folder in Drive → right-click → **Download** → upload the zip to the *Final mini-project* assignment in Google Classroom → **Turn in**. One submission per group, with every member's name in `PLAN.md`.

Before you do, check that **all five notebooks run top to bottom on a fresh runtime**, in order. If they only work in the session where you built them piece by piece, they do not yet reproduce — and 02 through 05 handing files to each other is exactly what makes that checkable.

In [ ]:
# Optional: build the bundle from here instead of a terminal.
# !cd .. && python scripts/make_submission.py --group $GROUP
